## Import tools needed for project

In [14]:
# Imports

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, ConfusionMatrixDisplay


## Cleaning file:  Review how many joint loans to consider remving from ananlysis

In [19]:
# Run check to see how many joint applications.  If small will eliminate those.

duckdb.query("""
    SELECT application_type, COUNT(*) as cnt
    FROM 'lending_club.csv'
    WHERE loan_status IN ('Charged Off', 'Fully Paid')
    GROUP BY application_type
""").df()

# Removed joint loans because they make up 1.78% of total loans in 'Charged off' and 'Fully Paid'


,application_type,cnt
0,Individual,1280370
1,Joint App,23237


In [20]:
# review loan_status categories
# Analyzing only loans that have paid or CO, no current loans

duckdb.query('''
    SELECT loan_status, COUNT(*) as cnt
    FROM "lending_club.csv"
    --WHERE loan_status IN ('Charged Off', 'Fully Paid')
    GROUP BY loan_status
''').df()

,loan_status,cnt
0,Current,919695
1,Default,31
2,Does not meet the credit policy. Status:Fully ...,1988
3,Late (16-30 days),3737
4,Does not meet the credit policy. Status:Charge...,761
5,Fully Paid,1041952
6,Late (31-120 days),21897
7,In Grace Period,8952
8,Charged Off,261655


## Select columns to use and filter by 'Charged Off' and 'Fully Paid'

In [25]:
# Filter to resolved loans only and select updated relevant columns

df = duckdb.query('''
SELECT  acc_open_past_24mths,
        addr_state,
        annual_inc,
        application_type,
        bc_util,
        collections_12_mths_ex_med,
        delinq_2yrs,
        dti,
        earliest_cr_line,
        emp_length,
        funded_amnt,
        grade,
        home_ownership,
        inq_last_6mths,
        int_rate,
        issue_d,
        loan_amnt,
        loan_status,
        mort_acc,
        mths_since_last_delinq,
        mths_since_last_major_derog,
        num_accts_ever_120_pd,
        num_op_rev_tl,
        num_tl_op_past_12m,
        open_acc,
        open_acc_6m,
        pct_tl_nvr_dlq,
        pub_rec,
        pub_rec_bankruptcies,
        purpose,
        revol_bal,
        revol_util,
        sub_grade,
        term,
        total_acc,
        verification_status
FROM 'lending_club.csv'
WHERE loan_status IN ('Charged Off', 'Fully Paid')
AND application_type = 'Individual'
''').df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Checking shape and memory usage --> repeat this after change made to review memory usage

In [26]:
print(df.shape)
print(df.memory_usage(deep=True).sum() / 1e6, "MB")

(1280370, 36)
499.783366 MB


In [35]:
# df = df.drop(columns=['application_type'])

In [36]:
print(df.shape)
# df.info()

(1280370, 35)


## Review dataframe for dtype, uniques, min, and max.  To decide which columns to reduce

In [52]:
# See all columns, dtypes, unique, min, and max values

summary = pd.DataFrame({
    'dtype': df.dtypes,
    'unique_values': df.nunique(),
    'min_values': df.min(),
    'max_values': df.max()
})
print(summary)

                               dtype  unique_values    min_values  max_values
acc_open_past_24mths            Int8             54             0          64
addr_state                       str             51            AK          WY
annual_inc                   float64          62057        2400.0  10999200.0
bc_util                      float64           1435           0.0       339.6
collections_12_mths_ex_med      Int8             15             0          20
delinq_2yrs                     Int8             30             0          39
dti                          float64           4787          -1.0       49.96
earliest_cr_line                 str            737      Apr-1934    Sep-2015
emp_length                       str             12        1 year         n/a
funded_amnt                    int64           1547           500       40000
grade                            str              7             A           G
home_ownership                   str              6           AN

## Converting str column 'term' to an Int8 dtype

In [46]:
# for 'term', replace months with '' and strip spaces, Int8

df['term'] = df['term'].str.replace('months', '').str.strip().astype('Int8')

In [47]:
df['term'].unique()

<IntegerArray>
[36, 60]
Length: 2, dtype: Int8

## Converting Int64 dtypes to Int8 --> reduces memory usage

In [51]:
# Int8: -128 to 127
# Capital 'I' instead of lower case 'i' in int allows nulls

int8_cols = ['acc_open_past_24mths', 'collections_12_mths_ex_med', 'delinq_2yrs', 'inq_last_6mths',
             'mort_acc', 'num_accts_ever_120_pd', 'num_op_rev_tl', 'num_tl_op_past_12m', 'open_acc_6m',
             'pub_rec', 'pub_rec_bankruptcies', 'open_acc']

for col in int8_cols:
    df[col] = df[col].astype('Int8')

In [59]:
# check Int8 columns for accuracy 

df[['acc_open_past_24mths', 'collections_12_mths_ex_med', 'delinq_2yrs', 'inq_last_6mths',
             'mort_acc', 'num_accts_ever_120_pd', 'num_op_rev_tl', 'num_tl_op_past_12m', 'open_acc_6m',
             'pub_rec', 'pub_rec_bankruptcies', 'open_acc', 'term']].info()

<class 'pandas.DataFrame'>
RangeIndex: 1280370 entries, 0 to 1280369
Data columns (total 13 columns):
 #   Column                      Non-Null Count    Dtype
---  ------                      --------------    -----
 0   acc_open_past_24mths        1233089 non-null  Int8 
 1   collections_12_mths_ex_med  1280314 non-null  Int8 
 2   delinq_2yrs                 1280370 non-null  Int8 
 3   inq_last_6mths              1280369 non-null  Int8 
 4   mort_acc                    1233089 non-null  Int8 
 5   num_accts_ever_120_pd       1212843 non-null  Int8 
 6   num_op_rev_tl               1212843 non-null  Int8 
 7   num_tl_op_past_12m          1212843 non-null  Int8 
 8   open_acc_6m                 476580 non-null   Int8 
 9   pub_rec                     1280370 non-null  Int8 
 10  pub_rec_bankruptcies        1279673 non-null  Int8 
 11  open_acc                    1280370 non-null  Int8 
 12  term                        1280370 non-null  Int8 
dtypes: Int8(13)
memory usage: 31.7 MB


## Converting Int64 dtypes to Int16 --> reduces memory usage

In [55]:
# Int16: -32,768 to 32767
# Capital 'I' instead of lower case 'i' in int allows nulls

int16_cols = ['mths_since_last_delinq', 'mths_since_last_major_derog', 'total_acc']

for col in int16_cols:
    df[col] = df[col].astype('Int16')

In [57]:
# check Int16 columns for accuracy 

df[['mths_since_last_delinq', 'mths_since_last_major_derog', 'total_acc']].info()

<class 'pandas.DataFrame'>
RangeIndex: 1280370 entries, 0 to 1280369
Data columns (total 3 columns):
 #   Column                       Non-Null Count    Dtype
---  ------                       --------------    -----
 0   mths_since_last_delinq       634569 non-null   Int16
 1   mths_since_last_major_derog  336801 non-null   Int16
 2   total_acc                    1280370 non-null  Int16
dtypes: Int16(3)
memory usage: 11.0 MB


## Converting Int64 dtypes to Int32 --> reduces memory usage

In [56]:
# Int32: up to ~2.1 billion
# Capital 'I' instead of lower case 'i' in int allows nulls

int32_cols = ['loan_amnt','revol_bal', 'funded_amnt']

for col in int32_cols:
    df[col] = df[col].astype('Int32')

In [58]:
# check Int32 columns for accuracy 

df[['loan_amnt','revol_bal', 'funded_amnt']].info()

<class 'pandas.DataFrame'>
RangeIndex: 1280370 entries, 0 to 1280369
Data columns (total 3 columns):
 #   Column       Non-Null Count    Dtype
---  ------       --------------    -----
 0   loan_amnt    1280370 non-null  Int32
 1   revol_bal    1280370 non-null  Int32
 2   funded_amnt  1280370 non-null  Int32
dtypes: Int32(3)
memory usage: 18.3 MB


In [65]:
print(df.shape)
print(df.memory_usage(deep=True).sum() / 1e6, "MB")

(1280370, 35)
296.204536 MB


## Converting float64 dtypes to Float32 --> reduces memory usage

In [61]:
# float32: +/- 3.4 x 10**38, precision: 7 decimal digits
# float64: +/- 1.8 x 10**308, precision: 15 decimal digits

# Capital 'F' instead of lower case 'f' in int allows nulls

float32_cols = ['annual_inc', 'bc_util', 'dti', 'int_rate', 'pct_tl_nvr_dlq', 'revol_util']

for col in float32_cols:
    df[col] = df[col].astype('Float32')

In [82]:
# Review Float32 cols for accuracy

df[['annual_inc', 'bc_util', 'dti', 'int_rate', 'pct_tl_nvr_dlq', 'revol_util']].info()

<class 'pandas.DataFrame'>
RangeIndex: 1280370 entries, 0 to 1280369
Data columns (total 6 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   annual_inc      1280370 non-null  Float32
 1   bc_util         1219530 non-null  Float32
 2   dti             1280370 non-null  Float32
 3   int_rate        1280370 non-null  Float32
 4   pct_tl_nvr_dlq  1212689 non-null  Float32
 5   revol_util      1279594 non-null  Float32
dtypes: Float32(6)
memory usage: 36.6 MB


## Covert 'str' columns to 'category' --> reduces memory usage:

##### The dtype 'category' is used when there are repeated values in a column.  Pandas stores these unique strings in a lookup table and represents rows using tiny underlying integers


In [73]:
# Review str columns for potential dtype 'category' assignment

df[['addr_state', 'earliest_cr_line', 'emp_length', 'grade', 'home_ownership', 'issue_d',
    'loan_status', 'purpose', 'sub_grade', 'verification_status']].nunique()

addr_state              51
earliest_cr_line       737
emp_length              12
grade                    7
home_ownership           6
issue_d                139
loan_status              2
purpose                 14
sub_grade               35
verification_status      3
dtype: int64

In [75]:
# Convert cols to dtype 'category'

cat_cols = ['addr_state', 'earliest_cr_line', 'emp_length', 'grade', 'home_ownership', 'issue_d',
            'loan_status', 'purpose', 'sub_grade', 'verification_status']

for col in cat_cols:
    df[col] = df[col].astype('category')

In [76]:
# Review cat cols for accuracy

df[['addr_state', 'earliest_cr_line', 'emp_length', 'grade', 'home_ownership', 'issue_d',
            'loan_status', 'purpose', 'sub_grade', 'verification_status']].info()

<class 'pandas.DataFrame'>
RangeIndex: 1280370 entries, 0 to 1280369
Data columns (total 10 columns):
 #   Column               Non-Null Count    Dtype   
---  ------               --------------    -----   
 0   addr_state           1280370 non-null  category
 1   earliest_cr_line     1280370 non-null  category
 2   emp_length           1280370 non-null  category
 3   grade                1280370 non-null  category
 4   home_ownership       1280370 non-null  category
 5   issue_d              1280370 non-null  category
 6   loan_status          1280370 non-null  category
 7   purpose              1280370 non-null  category
 8   sub_grade            1280370 non-null  category
 9   verification_status  1280370 non-null  category
dtypes: category(10)
memory usage: 14.7 MB


In [79]:
print(df.shape)
print(df.memory_usage(deep=True).sum() / 1e6, "MB")

(1280370, 36)
120.370692 MB


## Create target variable: 'default'

In [78]:
# Create target variable
df['default'] = (df['loan_status'] == 'Charged Off').astype('Int8')

## Save cleaned file to parquet

In [81]:
# Save to parquet

df.to_parquet('lending_club_clean.parquet', index=False, engine='fastparquet')

print('Saved successfully')
print('Final shape:', df.shape)
print('Final memory:', df.memory_usage(deep=True).sum() / 1e6, 'MB')

Saved successfully
Final shape: (1280370, 36)
Final memory: 120.370692 MB
